## Учимся получать эмбеддинги из DINOv2*

*Meta признана экстремистской организацией.

1. Рассчитываем cls_token dinov2 для всех ббоксов, полученных с помощью YOLO
2. Складываем их в ChromaDB
3. Папку ChromaDB кладём в s3

In [1]:
from tqdm import tqdm
from pathlib import Path
import os
from PIL import Image
import chromadb

from sneakers_hse.inference.embedding_model import DINOv2Embedder
from sneakers_hse.data.utils.s3_tools import S3Client

/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
PROJECT_ROOT_PATH = Path('/Users/a.r.makarenko/Documents/hse/sneakers-hse')
PATH_TO_YOLO_OUTPUT = PROJECT_ROOT_PATH / 'data/03_yolo_preprocessed/search-dataset-images'

# соберём все нужные файлы заранее
all_files = []
for root, _, files in os.walk(PATH_TO_YOLO_OUTPUT):
    for file in files:
        local_path = os.path.join(root, file)
        relative_path = os.path.relpath(local_path, PATH_TO_YOLO_OUTPUT)
        if not relative_path.endswith(".DS_Store"):
            all_files.append((root, file, relative_path))

len(all_files)

11301

In [3]:
# Инициализируем эмбеддер и векторную БД
embedder = DINOv2Embedder(device='mps')

client = chromadb.PersistentClient(
    path=PROJECT_ROOT_PATH / "chroma_db"
)
collection = client.get_or_create_collection(
    "embeddings",
    metadata={"hnsw:space": "cosine"})


W0504 08:24:09.794000 38871 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [4]:
def embed_bbox(relative_paths: list[str | Path]):
    image_paths = [PATH_TO_YOLO_OUTPUT / relative_path for relative_path in relative_paths]
    imgs = [Image.open(path) for path in image_paths]
    embeddings = embedder.encode_batch(imgs)
    ids = [str(relative_path) for relative_path in relative_paths]
    paths = relative_paths
    classes = [str(relative_path).split("/")[0] for relative_path in relative_paths]
    collection.add(
        ids=ids,
        embeddings=embeddings.tolist(),
        metadatas=[{
            "path": str(relative_path),
            "class": str(relative_path).split("/")[0]
        } for relative_path in relative_paths])
    return [{
                "path": path,
                "class": cls,
                "embedding": emb.tolist()
            } for emb, path, cls in zip(embeddings, paths, classes)]

embed_bbox(['adidas Кеды VL COURT 3/clothing_0_2.jpeg',
            'Nike Кеды Dunk Low Retro/clothing_0_86.jpeg'])

[{'path': 'adidas Кеды VL COURT 3/clothing_0_2.jpeg',
  'class': 'adidas Кеды VL COURT 3',
  'embedding': [-2.035252809524536,
   -0.4340837299823761,
   -1.3117990493774414,
   -0.12984734773635864,
   1.1193252801895142,
   -2.183021068572998,
   -1.493182897567749,
   -1.7145215272903442,
   2.8242955207824707,
   0.1449788510799408,
   0.4100984036922455,
   0.6773132681846619,
   -1.8693498373031616,
   0.14006152749061584,
   -2.0305867195129395,
   0.4253408908843994,
   -1.0870195627212524,
   -0.9815073013305664,
   -1.367919921875,
   0.9092010259628296,
   -2.7553305625915527,
   0.5979297757148743,
   4.450528621673584,
   0.9012638926506042,
   0.856468915939331,
   -2.1782639026641846,
   -0.5697842836380005,
   -1.8345214128494263,
   -1.591174602508545,
   2.67838191986084,
   -1.4795323610305786,
   -1.8476088047027588,
   1.2401405572891235,
   -1.038611888885498,
   -5.5558319091796875,
   -0.22303175926208496,
   -1.4060101509094238,
   -2.4640471935272217,
   0.503

In [5]:
def chunked(iterable, batch_size):
    batch = []
    for item in iterable:
        batch.append(item[2])
        if len(batch) == batch_size:
            yield batch
            batch = []
    if batch:
        yield batch

all_rows = []
# прогресс‑бар с полосой
for relative_paths in tqdm(chunked(iter(all_files), 32), desc="Processing images"):
    all_rows += embed_bbox(relative_paths)

Processing images: 22it [00:30,  1.38s/it]


KeyboardInterrupt: 

In [6]:
import polars as pl

df = pl.DataFrame(all_rows)
df.write_parquet(PROJECT_ROOT_PATH / 'data/04_dinov2_embeddings.parquet.gzip')

In [7]:
from dotenv import load_dotenv

load_dotenv()
s3 = S3Client(aws_access_key_id=os.getenv("AWS_ACCESS_KEY"),
              aws_secret_access_key=os.getenv("AWS_SECRET_KEY"))
s3.upload_folder_to_s3_parallel(
    bucket_name='sneakers-hse-images-test',
    s3_prefix='dinov2/chroma_db', # Путь пишем с учётом модели, т.е. каждой модели будет соответстовать отдельный инстанс ChromaDB
    local_folder=str(PROJECT_ROOT_PATH / 'chroma_db'),
    max_workers=10
)

Total files: 5
Uploaded: dinov2/chroma_db/a44b49bf-bdb0-4ddd-be29-5418ddff8a86/header.bin
Uploaded: dinov2/chroma_db/a44b49bf-bdb0-4ddd-be29-5418ddff8a86/link_lists.bin
Uploaded: dinov2/chroma_db/a44b49bf-bdb0-4ddd-be29-5418ddff8a86/length.bin
Uploaded: dinov2/chroma_db/a44b49bf-bdb0-4ddd-be29-5418ddff8a86/data_level0.bin
Uploaded: dinov2/chroma_db/chroma.sqlite3


### Проверяю, что query работает

In [8]:
image_path = PATH_TO_YOLO_OUTPUT / 'adidas Кеды VL COURT 3/clothing_0_2.jpeg'
img = Image.open(image_path)
embedding = embedder.encode_batch([img])
collection.query(embedding)

{'ids': [['adidas Кеды VL COURT 3/clothing_0_2.jpeg',
   'Vans Кеды Upland/clothing_0_212.jpeg',
   'Vans Кеды Upland/orig_108.jpeg',
   'X-Plode Кеды/orig_124.jpeg',
   'X-Plode Кеды/orig_35.jpeg',
   'Vans Кеды Upland/clothing_0_36.jpeg',
   'Vans Кеды Upland/orig_36.jpeg',
   'Vans Кеды Upland/clothing_0_132.jpeg',
   'X-Plode Кеды/orig_165.jpeg',
   'Vans Кеды Upland/orig_133.jpeg']],
 'embeddings': None,
 'documents': [[None, None, None, None, None, None, None, None, None, None]],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'class': 'adidas Кеды VL COURT 3',
    'path': 'adidas Кеды VL COURT 3/clothing_0_2.jpeg'},
   {'path': 'Vans Кеды Upland/clothing_0_212.jpeg',
    'class': 'Vans Кеды Upland'},
   {'path': 'Vans Кеды Upland/orig_108.jpeg', 'class': 'Vans Кеды Upland'},
   {'path': 'X-Plode Кеды/orig_124.jpeg', 'class': 'X-Plode Кеды'},
   {'class': 'X-Plode Кеды', 'path': 'X-Plode Кеды/orig_35.jpeg'},
   {'path': 'Vans К

In [ ]:
# Проверяю, что и без нормализации косинусное расстояние корректно считается
import numpy as np


a = embedding[0]
b = collection.get('Vans Кеды Upland/clothing_0_212.jpeg', include=['embeddings'])['embeddings'][0]
1 - np.dot(a, b) / np.linalg.norm(a) / np.linalg.norm(b) 

np.float64(0.09111805732035672)

In [20]:
b = collection.get('Vans Кеды Upland/orig_108.jpeg', include=['embeddings'])['embeddings'][0]
1 - np.dot(a, b) / np.linalg.norm(a) / np.linalg.norm(b) 

np.float64(0.11358863712257539)